In [ ]:
# terminal
# XXX is the spectrograms_approach2.zip id from google drive
# YYY is the genres_metadata.zip
apt-get update && apt-get install -y ffmpeg
apt-get update && apt-get install -y unzip

pip install gdown

gdown --id XXX
gdown --id YYY

unzip spectrograms_approach2.zip
unzip genres_metadata.zip

In [ ]:
# terminal
# may need to run this if cannot install unzip
apt-get update -o Dir::Etc::sourcelist="sources.list.d/ubuntu.sources" -o Dir::Etc::sourceparts="-" -o APT::Get::List-Cleanup="0"
apt-get install -y unzip

In [ ]:
!pip install librosa
!pip install pydub
!pip install pandas
!pip install torchmetrics
!pip install matplotlib

In [ ]:
import sys
!git clone https://github.com/majfu/MusicGenreClassifier.git
repo_path = "/workspace/MusicGenreClassifier"
sys.path.append(repo_path)
sys.path.insert(0, repo_path)

In [ ]:
import os

GENRES_METADATA_DIR = "/workspace/genres_metadata"
MODELS_DIRECTORY_RP = "/workspace/models/"
# as GENRE set the name of one of the genre folders from genres_metadata.zip, e.g. :
GENRE = "Classical"
DATA_DIR = GENRES_METADATA_DIR + "/" + GENRE

MEAN_TRAIN_RP = os.path.join(DATA_DIR, "train_mean.pt")
STD_TRAIN_RP = os.path.join(DATA_DIR, "train_std.pt")
MEAN_VAL_RP = os.path.join(DATA_DIR, "val_mean.pt")
STD_VAL_RP = os.path.join(DATA_DIR, "val_std.pt")

TRAIN_SPLIT_RP = os.path.join(DATA_DIR, "train.csv")
VAL_SPLIT_RP = os.path.join(DATA_DIR, "val.csv")

SPECTROGRAMS_RP = "/workspace/spectrograms_approach2"
PLOTS_RP = "/workspace/plots" + "/" + GENRE

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from src.features.standardization import StandardizationTransform
from src.data.feature_dataset import FeatureDataset
from src.utils.io_utils import *
from src.utils.metadata_utils import *
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall
import matplotlib.pyplot as plt

In [ ]:
from src.data.feature_dataset import FeatureDataset
from src.utils.io_utils import calculate_and_save_dataset_mean_and_std

train_fds = FeatureDataset(TRAIN_SPLIT_RP, SPECTROGRAMS_RP)
val_fds = FeatureDataset(VAL_SPLIT_RP, SPECTROGRAMS_RP)

# after computing them all download them onto permanent disk
calculate_and_save_dataset_mean_and_std(train_fds, MEAN_TRAIN_RP, STD_TRAIN_RP)
calculate_and_save_dataset_mean_and_std(val_fds, MEAN_VAL_RP, STD_VAL_RP)

In [ ]:
# for Pop the result is better when dropout is removed from convolutional layers
class MusicGenreCNN(nn.Module):
    def __init__(self, hidden_size=512):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3, 3), padding='same', stride=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        self.conv2 = nn.Sequential(
            nn.Dropout(0.2),
            nn.Conv2d(32, 64, kernel_size=(3, 3), padding='same', stride=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        self.conv3 = nn.Sequential(
            nn.Dropout(0.2),
            nn.Conv2d(64, 128, kernel_size=(3, 3), padding='same', stride=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        self.conv4 = nn.Sequential(
            nn.Dropout(0.2),
            nn.Conv2d(128, 256, kernel_size=(3, 3), padding='same', stride=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.flatten = nn.Flatten()
        self.average_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(256, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.average_pooling(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def train(model, num_epochs, train_dl, val_dl, version_num, early_stopping_patience=10, min_loss_delta=0.003):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    val_accuracy = BinaryAccuracy().to(DEVICE)
    val_precision = BinaryPrecision().to(DEVICE)
    val_recall = BinaryRecall().to(DEVICE)
    
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_precision": [],
        "val_recall": []
    }

    scaler = GradScaler(enabled=True)

    best_val_loss = float("inf")
    best_epoch = -1
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for x_batch, y_batch in train_dl:
            x_batch = x_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.float().to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            x_batch = x_batch.unsqueeze(1).contiguous(memory_format=torch.channels_last)

            with autocast(device_type="cuda", dtype=USE_AMP_DTYPE):
                logits = model(x_batch)
                loss = criterion(logits, y_batch)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        val_accuracy.reset(); val_precision.reset(); val_recall.reset()

        for x_batch, y_batch in val_dl:
            x_batch = x_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.float().to(DEVICE, non_blocking=True)
            x_batch = x_batch.unsqueeze(1).contiguous(memory_format=torch.channels_last)

            with autocast(device_type="cuda", dtype=USE_AMP_DTYPE):
                logits = model(x_batch)
                batch_loss = criterion(logits, y_batch)

            val_loss += batch_loss.item()
            preds = torch.sigmoid(logits)
            val_accuracy.update(preds, y_batch)
            val_precision.update(preds, y_batch)
            val_recall.update(preds, y_batch)

        avg_val_loss = val_loss / len(val_dl)
        avg_train_loss = train_loss / len(train_dl)
        scheduler.step(avg_val_loss)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)

        history["val_accuracy"].append(val_accuracy.compute().item())
        history["val_precision"].append(val_precision.compute().item())
        history["val_recall"].append(val_recall.compute().item())

        print(f"Epoch {epoch + 1}, train loss: {avg_train_loss:.4f}, val loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss - min_loss_delta:
            best_val_loss = avg_val_loss
            best_epoch = epoch
            patience_counter = 0

            best_model_path = os.path.join(MODELS_DIRECTORY_RP, f'{GENRE}_version_{version_num}_best.pt')
            torch.save(model.state_dict(), best_model_path)
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}, best val loss {best_val_loss:.4f}")
                for key in history.keys():
                    history[key] = history[key][: best_epoch + 1]
                break


    return history

In [ ]:
def plot_and_save_history(history, version_num, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    epochs = range(len(history["val_loss"]))

    # Loss
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Training vs Validation Loss")
    plt.savefig(os.path.join(output_dir, f"loss_version{version_num}.png"))
    plt.close()

    # F1 Score
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.title("Validation Accuracy")
    plt.savefig(os.path.join(output_dir, f"accuracy_version{version_num}.png"))
    plt.close()

    # Precision
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_precision"], label="Validation Precision")
    plt.xlabel("Epochs")
    plt.ylabel("Precision")
    plt.legend()
    plt.title("Validation Precision")
    plt.savefig(os.path.join(output_dir, f"precision_version{version_num}.png"))
    plt.close()

    # Recall
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_recall"], label="Validation Recall")
    plt.xlabel("Epochs")
    plt.ylabel("Recall")
    plt.legend()
    plt.title("Validation Recall")
    plt.savefig(os.path.join(output_dir, f"recall_version{version_num}.png"))
    plt.close()

    print(f"Training history plots saved to '{output_dir}'")


In [ ]:
train_mean = torch.load(MEAN_TRAIN_RP).float()
train_std = torch.load(STD_TRAIN_RP).float()
val_mean = torch.load(MEAN_VAL_RP).float()
val_std = torch.load(STD_VAL_RP).float()

train_transform = StandardizationTransform(train_mean, train_std)
val_transform = StandardizationTransform(val_mean, val_std)

standardized_training_dataset = FeatureDataset(
    TRAIN_SPLIT_RP,
    SPECTROGRAMS_RP,
    transform=train_transform
)
standardized_val_dataset = FeatureDataset(
    VAL_SPLIT_RP,
    SPECTROGRAMS_RP,
    transform=val_transform
)

train_labels_df = load_encoded_labels_df(TRAIN_SPLIT_RP).drop('track_id', axis=1)

model = MusicGenreCNN()
model = model.to(DEVICE, memory_format=torch.channels_last)
try:
    model = torch.compile(model)
except Exception:
    pass

train_dl = DataLoader(
    standardized_training_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=10,
    pin_memory=True if DEVICE.type == "cuda" else False,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=True
)
val_dl = DataLoader(
    standardized_val_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=10,
    pin_memory=True if DEVICE.type == "cuda" else False,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=False
)

if DEVICE.type != "cuda":
    print("CUDA not detected.")

os.makedirs(MODELS_DIRECTORY_RP, exist_ok=True)

version_num = 1
epochs_num = 200
history = train(model, epochs_num, train_dl, val_dl, version_num)
plot_and_save_history(history, version_num, PLOTS_RP)